# Task 1: Named Entity Recognition (NER) for Mountain Extraction

This notebook demonstrates the demo and evaluation of our fine-tuned `distilbert-base-uncased` model trained to identify mountain names within arbitrary text.

### Features tested:
- Standard positive examples (famous mountains, ranges).
- Negative examples / edge cases (software names like *Ushba*, exception terms like *Nun*, RAM specs like *Cerro Plata*).

In [4]:
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# local path where trained weights are saved
model_path = "./mountain_ner_model"

# if running in colab and model folder is missing, download from gdrive via gdown
if not os.path.exists(model_path):
    import gdown
    file_id = "1ejL3T1l7bc1hYXNogzhHSkbh---H-EJZ"
    gdown.download(id=file_id, output="mountain_ner_model.zip", quiet=False)
    !unzip mountain_ner_model.zip

# load tokenizer and fine-tuned distilbert model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6935.77it/s]


In [5]:
# initialize huggingface ner pipeline with simple aggregation to merge b- and i- tokens
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

## Evaluation on Diverse Test Cases

We evaluate the model using a curated set of test sentences including positive contexts (real geographical entities) and challenging negative contexts (contextual homonyms).

In [ ]:
# predefined test cases covering both clear positive mentions and negative/app edge cases
test_sentences = [
    # positive cases
    "The climbers reached the summit of Everest at dawn.",
    "K2 is considered one of the most dangerous places in the world.",
    "Meltwater from Lanín feeds several river networks below.",
    "Local indigenous communities hold deep cultural reverence for Wollumbin / Mount Warning.",
    
    # negative cases (should not classify app/code terms as mountains)
    "I just updated my Ushba app and it keeps crashing.",
    "Can someone explain why Nun keeps throwing a null pointer exception?",
    "According to the documentation, Cerro Plata requires 16GB of RAM."
]

results = []

# process sentences and extract results
for text in test_sentences:
    preds = ner_pipeline(text)
    mountains = [p["word"].strip() for p in preds if p.get("entity_group") == "MOUNTAIN"]
    
    results.append({
        "Input Sentence": text,
        "Detected Entities": ", ".join(mountains) if mountains else "None",
        "Raw Output": [(p["word"], round(p["score"], 3)) for p in preds]
    })

# display summary table in notebook
df_results = pd.DataFrame(results)
df_results

,Input Sentence,Detected Entities,Raw Output
0,The climbers reached the summit of Everest at ...,everest,"[(everest, 0.995)]"
1,K2 is considered one of the most dangerous pla...,k2,"[(k2, 0.994)]"
2,Meltwater from Lanín feeds several river netwo...,lanin,"[(lanin, 0.999)]"
3,Local indigenous communities hold deep cultura...,wollumbin / mount warning.,"[(wollumbin / mount warning., 0.998)]"
4,I just updated my Ushba app and it keeps crash...,None,[]
5,Can someone explain why Nun keeps throwing a n...,None,[]
6,"According to the documentation, Cerro Plata re...",None,[]
